In [ ]:
from pathlib import Path
import gc
import os
import random

def absolute(path):
    return Path(os.path.abspath(os.fspath(path)))

ROOT = absolute(Path.cwd())
if ROOT.name == 'src':
    ROOT = ROOT.parent
WORK_DIR = ROOT.parent / 'Code_run'

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

BRANCHES = ['county_holdout', 'utility_holdout', 'full_sample_fit']
SELECTED_SEASONS = ['spring', 'summer', 'fall', 'winter']

In [ ]:
LABEL = 'ELEC_M'
FEATURES = [
    'ELEP_M', 'LST_HDD', 'LST_CDD', 'LST_STD', 'NTL', 'PM25', 'NDVI', 'NDBI',
    'EVI', 'LONGITUDE', 'LATITUDE', 'POP_DENS', 'PCT_DEM', 'AC_RECS',
    'ACS_MEDIAN_HH_INC', 'ACS_PCT_HU_ELEC', 'ACS_PCT_HU_SOLAR',
    'ACS_PCT_OWNER_HU_COST_30PCT', 'ACS_PCT_RENTER_HU', 'ACS_MEDIAN_RENT',
    'ACS_MEDIAN_HOME_VALUE', 'ACS_PCT_COMMT_60MINUP',
    'ACS_PCT_BLACK_NONHISP', 'ACS_PCT_HISPANIC',
]
SEASONS = {'spring': [3, 4, 5], 'summer': [6, 7, 8],
           'fall': [9, 10, 11], 'winter': [1, 2, 12]}
SEED, N_FOLDS = 3407, 8
FIT_SETTINGS = dict(presets='best_quality', dynamic_stacking=False,
                    num_bag_folds=8, num_bag_sets=1, num_stack_levels=1,
                    num_cpus=32, num_gpus=0)
UTILITY_SETTINGS = dict(fit_strategy='sequential',
                        ag_args_ensemble={'fold_fitting_strategy': 'sequential_local'})
REFIT_SETTINGS = dict(model='best', set_best_to_refit_full=True,
                      num_cpus=32, num_gpus=0)
PREDICTION_BATCH_SIZE = 100_000

def county_folds(counties):
    ids = np.asarray(sorted(set(counties)), dtype=np.int64)
    np.random.RandomState(SEED).shuffle(ids)
    return {int(county): fold for fold, group in enumerate(np.array_split(ids, N_FOLDS))
            for county in group}

def thermal_features(frame):
    frame = frame.copy()
    days = pd.to_datetime({'year': frame.YEAR, 'month': frame.MONTH,
                           'day': 1}).dt.days_in_month.to_numpy(float)
    lst = frame.LST.to_numpy(float)
    frame['LST_HDD'] = days * np.maximum(65.0 - lst, 0.0)
    frame['LST_CDD'] = days * np.maximum(lst - 65.0, 0.0)
    return frame

def performance(actual, predicted):
    from sklearn.metrics import r2_score
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    residual = predicted - actual
    return {'n': len(actual), 'r2': float(r2_score(actual, predicted)),
            'rmse': float(np.sqrt(np.mean(residual ** 2))),
            'mape_percent': float(np.mean(np.abs(residual / actual)) * 100),
            'mae': float(np.mean(np.abs(residual))), 'bias': float(np.mean(residual))}

In [ ]:
def training_data(season, branch):
    frame = pd.read_parquet(ROOT / 'data/training' / f'{season}.parquet')
    frame = frame.sort_values(['GEOID', 'MONTH']).reset_index(drop=True)
    if branch == 'utility_holdout':
        frame['partition'] = frame.utility_partition
        frame['_county_fold'] = frame.utility_fold
    elif branch == 'full_sample_fit':
        frame['partition'] = 'train'
        frame['_county_fold'] = frame.GEOID.map(county_folds(frame.GEOID)).astype(int)
    train = frame.loc[frame.partition.eq('train')].copy()
    test = frame.loc[frame.partition.eq('test')].copy()
    return train, test

In [ ]:
def fit_one(branch, season):
    from autogluon.tabular import TabularPredictor
    import ray

    model_path = WORK_DIR / 'models' / branch / season
    if model_path.exists():
        raise FileExistsError(f'{model_path} already exists. Use a new WORK_DIR for a new fit.')
    train, test = training_data(season, branch)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    settings = dict(FIT_SETTINGS)
    if branch == 'utility_holdout':
        settings.update(UTILITY_SETTINGS)
    random.seed(SEED)
    np.random.seed(SEED)
    predictor = TabularPredictor(label=LABEL, problem_type='regression',
        eval_metric='root_mean_squared_error', groups='_county_fold',
        path=str(model_path), verbosity=2, log_to_file=True,
        log_file_path=str(WORK_DIR / f'{branch}_{season}_training.log'))
    previous = Path.cwd()
    try:
        (model_path / 'models').mkdir(parents=True, exist_ok=True)
        os.chdir(model_path / 'models')
        predictor.fit(train_data=train[FEATURES + [LABEL, '_county_fold']],
                      **settings)
        if branch == 'full_sample_fit':
            os.chdir(model_path / 'models')
            predictor.refit_full(**REFIT_SETTINGS)
        predictor.save()
        if len(test):
            prediction = predictor.predict(test[FEATURES]).to_numpy()
            result = test[['GEOID', 'MONTH', LABEL]].copy()
            result['prediction'] = np.asarray(prediction, dtype=np.float64)
            result.to_csv(WORK_DIR / f'{branch}_{season}_predictions.csv', index=False)
            scores = performance(test[LABEL], prediction)
            pd.DataFrame([scores]).to_csv(WORK_DIR / f'{branch}_{season}_metrics.csv', index=False)
            print(branch, season, scores)
        else:
            print(branch, season, 'Saved full-sample model:', predictor.model_best)
    finally:
        os.chdir(previous)
        if ray.is_initialized():
            ray.shutdown()
        predictor.unpersist(models='all')
        del predictor
        gc.collect()

for branch in BRANCHES:
    for season in SELECTED_SEASONS:
        fit_one(branch, season)

In [ ]:
def predict_one(season):
    from autogluon.tabular import TabularPredictor
    model_path = WORK_DIR / 'models' / 'full_sample_fit' / season
    source = ROOT / 'data/prediction' / f'{season}.parquet'
    target = WORK_DIR / 'predictions' / f'{season}.parquet'
    temporary = target.with_suffix('.tmp.parquet')
    (WORK_DIR / 'predictions').mkdir(parents=True, exist_ok=True)
    keys = ['GEOID', 'YEAR', 'MONTH']
    columns = list(dict.fromkeys(keys + ['LST'] +
        [name for name in FEATURES if name not in ['LST_HDD', 'LST_CDD']]))
    schema = pa.schema([('GEOID', pa.int64()), ('YEAR', pa.int16()),
                        ('MONTH', pa.int8()), ('PRED_ELEC_M', pa.float64())])
    predictor = TabularPredictor.load(str(model_path), verbosity=0)
    try:
        predictor.persist(models='best', with_ancestors=True, max_memory=0.35)
        parquet = pq.ParquetFile(source)
        rows = 0
        with pq.ParquetWriter(temporary, schema=schema, compression='snappy') as writer:
            for batch in parquet.iter_batches(batch_size=PREDICTION_BATCH_SIZE,
                                              columns=columns, use_threads=True):
                frame = batch.to_pandas()
                for name in keys:
                    frame[name] = pd.to_numeric(frame[name], errors='raise').astype('int64')
                data = thermal_features(frame)
                prediction = predictor.predict(data[FEATURES],
                                               model=predictor.model_best).to_numpy(float)
                result = frame[keys].copy()
                result['PRED_ELEC_M'] = prediction
                writer.write_table(pa.Table.from_pandas(result, schema=schema,
                                                       preserve_index=False))
                rows += len(frame)
                print(season, rows, '/', parquet.metadata.num_rows)
        os.replace(temporary, target)
    finally:
        predictor.unpersist(models='all')
        del predictor
        gc.collect()

for season in SELECTED_SEASONS:
    predict_one(season)